In [3]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-optimization-drill").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 15:44:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


# Drill -- Movie Analytics with Spark SQL

The file `data/movies.json` contains data about movies (IMDB, Rotten Tomatoes, Box Office).

Using **Spark SQL** (create a temp view, then `spark.sql(...)`), answer the following:

1. How many movies are in the dataset?
2. How many **distinct** major genres? (column: `Major_Genre`)
3. What is the **approximate** count of distinct genres? (`approx_count_distinct`)
4. What are the **min** and **max** IMDB ratings? (column: `IMDB_Rating`)
5. What is the **total** US gross revenue? (column: `US_Gross`)
6. What is the **mean** and **standard deviation** of Rotten Tomatoes ratings? (`Rotten_Tomatoes_Rating`)
7. How many movies per genre? (group by `Major_Genre`, count, order by count descending)
8. Average IMDB rating per genre, ordered by average rating descending

Try to optimize the performance for the queries below.

In [4]:
movies = spark.read.option("inferSchema", "true").json("data/movies.json")
movies.createOrReplaceTempView("movies")
movies.printSchema()
movies.show(5)

root
 |-- Creative_Type: string (nullable = true)
 |-- Director: string (nullable = true)
 |-- Distributor: string (nullable = true)
 |-- IMDB_Rating: double (nullable = true)
 |-- IMDB_Votes: long (nullable = true)
 |-- MPAA_Rating: string (nullable = true)
 |-- Major_Genre: string (nullable = true)
 |-- Production_Budget: long (nullable = true)
 |-- Release_Date: string (nullable = true)
 |-- Rotten_Tomatoes_Rating: long (nullable = true)
 |-- Running_Time_min: long (nullable = true)
 |-- Source: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- US_DVD_Sales: long (nullable = true)
 |-- US_Gross: long (nullable = true)
 |-- Worldwide_Gross: long (nullable = true)

+--------------------+--------+-----------+-----------+----------+-----------+-----------+-----------------+------------+----------------------+----------------+-------------------+--------------------+------------+--------+---------------+
|       Creative_Type|Director|Distributor|IMDB_Rating|IMDB_Votes|M

In [5]:
# 1. Total count
spark.sql("SELECT COUNT(*) AS total_movies FROM movies").show()

+------------+
|total_movies|
+------------+
|        3201|
+------------+



In [6]:
# 2. Exact distinct genres
spark.sql("SELECT COUNT(DISTINCT Major_Genre) AS distinct_genres FROM movies").show()

+---------------+
|distinct_genres|
+---------------+
|             12|
+---------------+



In [7]:
# 3. Approximate distinct genres (faster for large datasets)
spark.sql("SELECT APPROX_COUNT_DISTINCT(Major_Genre) AS approx_distinct_genres FROM movies").show()

+----------------------+
|approx_distinct_genres|
+----------------------+
|                    12|
+----------------------+



26/06/12 15:44:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [8]:
# 4. Min and max IMDB rating
spark.sql("SELECT MIN(IMDB_Rating) AS min_rating, MAX(IMDB_Rating) AS max_rating FROM movies").show()

+----------+----------+
|min_rating|max_rating|
+----------+----------+
|       1.4|       9.2|
+----------+----------+



In [9]:
# 5. Total US gross revenue
spark.sql("SELECT SUM(US_Gross) AS total_us_gross FROM movies").show()

+--------------+
|total_us_gross|
+--------------+
|  140542660013|
+--------------+



In [10]:
# 6. Mean and stddev Rotten Tomatoes
spark.sql("SELECT ROUND(MEAN(Rotten_Tomatoes_Rating), 2) AS mean_rt, ROUND(STDDEV(Rotten_Tomatoes_Rating), 2) AS stddev_rt FROM movies").show()

+-------+---------+
|mean_rt|stddev_rt|
+-------+---------+
|  54.34|    28.08|
+-------+---------+



In [11]:
# 7. Count per genre
spark.sql("SELECT Major_Genre, COUNT(*) AS count FROM movies GROUP BY Major_Genre ORDER BY count DESC").show()

+-------------------+-----+
|        Major_Genre|count|
+-------------------+-----+
|              Drama|  789|
|             Comedy|  675|
|             Action|  420|
|               NULL|  275|
|          Adventure|  274|
|  Thriller/Suspense|  239|
|             Horror|  219|
|    Romantic Comedy|  137|
|            Musical|   53|
|        Documentary|   43|
|       Black Comedy|   36|
|            Western|   36|
|Concert/Performance|    5|
+-------------------+-----+



In [12]:
# 8. Average IMDB rating per genre
spark.sql("SELECT Major_Genre, COUNT(*) AS movie_count, ROUND(AVG(IMDB_Rating), 2) AS avg_imdb FROM movies GROUP BY Major_Genre ORDER BY avg_imdb DESC").show()

+-------------------+-----------+--------+
|        Major_Genre|movie_count|avg_imdb|
+-------------------+-----------+--------+
|        Documentary|         43|     7.0|
|            Western|         36|    6.84|
|       Black Comedy|         36|    6.82|
|              Drama|        789|    6.77|
|               NULL|        275|     6.5|
|            Musical|         53|    6.45|
|  Thriller/Suspense|        239|    6.36|
|          Adventure|        274|    6.35|
|Concert/Performance|          5|    6.33|
|             Action|        420|    6.11|
|    Romantic Comedy|        137|    5.87|
|             Comedy|        675|    5.85|
|             Horror|        219|    5.68|
+-------------------+-----------+--------+



In [13]:
spark.stop()